In [1]:
repo_url = "https://github.com/Anaemos/LeafGreen.git"
# clone
!git clone {repo_url}

Cloning into 'LeafGreen'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 23 (delta 5), reused 18 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 9.72 KiB | 9.72 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [2]:
%cd /content/LeafGreen

/content/LeafGreen


In [3]:
import getpass

token = getpass.getpass("Enter GitHub Token: ")

Enter GitHub Token: ··········


In [4]:
!git config --global user.email "aryavartsinghpayal@gmail.com"
!git config --global user.name "Anaemos"

In [5]:
import os

remote_with_token = f"https://{token}@github.com/Anaemos/LeafGreen.git"
!git remote set-url origin {remote_with_token}

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Data/processed.zip"
extract_path = "/content/processed"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)


Extracted to: /content/processed


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import os
from tqdm import tqdm

BASE_DIR = "/content/processed/processed"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

print(TRAIN_DIR, VAL_DIR, TEST_DIR)


/content/processed/processed/train /content/processed/processed/val /content/processed/processed/test


In [9]:
IMAGE_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

In [10]:
train_dataset = ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = ImageFolder(VAL_DIR, transform=val_test_transforms)
test_dataset  = ImageFolder(TEST_DIR, transform=val_test_transforms)

num_classes = len(train_dataset.classes)

print("Total classes:", num_classes)
print("Sample classes:", train_dataset.classes[:10])

Total classes: 38
Sample classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight']


In [12]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
len(train_loader), len(val_loader), len(test_loader)

(1358, 170, 171)

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet50(weights="IMAGENET1K_V2")
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [17]:
EPOCHS = 10
best_val_acc = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for imgs, labels in tqdm(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f}")

    # validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)

            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Val Acc: {val_acc:.4f}")

    # save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "leafgreen_resnet50.pth")
        print("New best model saved.")



Epoch 1/10


100%|██████████| 1358/1358 [08:06<00:00,  2.79it/s]


Train Loss: 0.2723 | Train Acc: 0.9291
Val Acc: 0.9891
New best model saved.

Epoch 2/10


100%|██████████| 1358/1358 [08:02<00:00,  2.81it/s]


Train Loss: 0.0411 | Train Acc: 0.9872
Val Acc: 0.9910
New best model saved.

Epoch 3/10


...

Train Loss: 0.0113 | Train Acc: 0.9970
Val Acc: 0.9954

Epoch 9/10


100%|██████████| 1358/1358 [08:06<00:00,  2.79it/s]


Train Loss: 0.0122 | Train Acc: 0.9964
Val Acc: 0.9956

Epoch 10/10


100%|██████████| 1358/1358 [08:04<00:00,  2.80it/s]


Train Loss: 0.0092 | Train Acc: 0.9971
Val Acc: 0.9947


In [18]:
model.load_state_dict(torch.load("leafgreen_resnet50.pth"))
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)

        _, predicted = outputs.max(1)
        test_correct += predicted.eq(labels).sum().item()
        test_total += labels.size(0)

test_acc = test_correct / test_total
print(f"\n FINAL TEST ACCURACY: {test_acc:.4f}")



 FINAL TEST ACCURACY: 0.9939


In [19]:
!cp leafgreen_resnet50.pth /content/drive/MyDrive/
print("Model saved to Drive!")

Model saved to Drive!
